# <u> Monthly Gym Pledge Analytics </u>

- Author: Srikar Gunisetty
- Last Modified Date: 01/03/2026

**<u> Objective:</u>** This notebook analyzes a Google Sheets log of workouts with the following fields:

- Timestamp: when the form response was submitted
- Name: who submitted
- Workout date: the date the workout occurred
- Burnt >= 250 calories?: a boolean (Yes/No, True/False) indicating whether the workout meets the 250+ calories threshold

**<u>Primary goal:</u>**
- Determine monthly winners (>= 16 unique workout days in a month where Burnt >= 250 calories is true)

**<u>Secondary goals:</u>**
- Track consistency (including workouts under 250 calories)
- Day-of-week patterns
- Logging delay (timestamp vs workout date)
- Front-loading vs end-of-month cramming
- Streaks and inconsistency
- "Barely missed" (e.g., 14–15 qualifying days)
- Month-over-month trends and gamification awards

## <u> Setup & Data Import</u>

In [5]:
# libraries
import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

import gspread
from google.oauth2.service_account import Credentials

#set plot theme
sns.set_theme(style = "darkgrid", palette = "muted")

#set plot preferences
mpl.rcParams["figure.dpi"] = 150
plt.rcParams["font.family"] = "Consolas"
%matplotlib inline

import warnings
warnings.simplefilter(action = "ignore", category = FutureWarning)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

In [6]:
# 1) Your spreadsheet id is the long string in the URL
SPREADSHEET_ID = "17RADj_LH-Lj_lB8QFZyxjv8iKFerNZfNT-7wmtPNuzk"

# 2) This must match your tab name exactly (bottom of the Google Sheet)
WORKSHEET_NAME = "Form Responses"

# 3) Path to the JSON key you downloaded
SERVICE_ACCOUNT_JSON_PATH = "secrets/service_account.json"

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets.readonly",
    "https://www.googleapis.com/auth/drive.readonly",
]

def read_google_sheet_as_df(spreadsheet_id: str, worksheet_name: str, json_path: str) -> pd.DataFrame:
    creds = Credentials.from_service_account_file(json_path, scopes=SCOPES)
    gc = gspread.authorize(creds)

    sh = gc.open_by_key(spreadsheet_id)
    ws = sh.worksheet(worksheet_name)

    values = ws.get_all_values()
    if not values:
        return pd.DataFrame()

    headers = values[0]
    rows = values[1:]
    df = pd.DataFrame(rows, columns=headers)

    df = df.replace("", pd.NA).dropna(how="all")
    return df

raw = read_google_sheet_as_df(SPREADSHEET_ID, WORKSHEET_NAME, SERVICE_ACCOUNT_JSON_PATH)
raw.head(10)

,Timestamp,You are?,Workout date,Burnt >= 250 calories?
0,1/1/2026 15:30:34,Naveen Ganta,1/1/2026,Yes
1,1/1/2026 18:45:05,Eshwar,1/1/2026,Yes
2,1/1/2026 19:56:13,Srikar Gunisetty,1/1/2026,Yes
3,1/1/2026 21:27:05,Sandesh Ghanta,1/1/2026,Yes
4,1/1/2026 21:59:46,Surya Chaitanya,1/1/2026,Yes
5,1/1/2026 22:11:58,Vennela Chava,1/1/2026,Yes
6,1/1/2026 23:10:27,Pradyumna Ch.,1/1/2026,Yes
7,1/2/2026 19:57:22,Srivatsav Gunisetty,1/2/2026,Yes
8,1/2/2026 20:13:34,Pradyumna Ch.,1/2/2026,Yes
9,1/2/2026 22:04:31,Surya Chaitanya,1/2/2026,Yes


## <u>Cleaning and normalization</u>

Key cleaning steps:
- Parse Timestamp and Workout date into proper datetime types
- Normalize names (trim spaces, consistent casing)
- Standardize the 250+ calories flag into a boolean
- Deduplicate submissions so a person only gets credit once per workout date

Notes on data semantics:
- Qualifying workout day: a unique workout_date where burnt_250 is true
- Any workout day: a unique workout_date regardless of burnt_250 (used for consistency and streaks)

In [8]:
import pandas as pd
import numpy as np

def normalize_bool(x) -> bool:
    if pd.isna(x):
        return False
    s = str(x).strip().lower()
    return s in {"yes", "true", "1", "y", "t"}

def clean(
    df: pd.DataFrame,
    *,
    col_timestamp: str = "Timestamp",
    col_name: str = "You are?",
    col_wkdate: str = "Workout date",
    col_250: str = "Burnt >= 250 calories?",
    dedupe: bool = True,
) -> pd.DataFrame:
    df = df.copy()

    # if expected columns are missing
    expected = {col_timestamp, col_name, col_wkdate, col_250}
    missing = expected - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}. Found columns: {list(df.columns)}")

    # Rename for internal consistency
    df = df.rename(columns={
        col_timestamp: "timestamp",
        col_name: "name",
        col_wkdate: "workout_date",
        col_250: "burnt_250_raw",
    })

    # Parse datetimes
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

    # Workout date can be missing (NA) or a string; store as date
    wk = pd.to_datetime(df["workout_date"], errors="coerce")
    df["workout_date"] = wk.dt.date

    # Normalize name
    df["name_raw"] = df["name"]
    df["name"] = (
        df["name"]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    # Standardize boolean
    df["burnt_250"] = df["burnt_250_raw"].apply(normalize_bool)

    # Any workout means workout_date exists
    df["any_workout"] = df["workout_date"].notna()

    # Optional dedupe: one credit per person per workout day (earliest submission wins)
    if dedupe:
        df = df.sort_values(["name", "workout_date", "timestamp"])
        df = df.drop_duplicates(subset=["name", "workout_date"], keep="first")

    # Derived time columns (safe even when workout_date is missing)
    df["workout_dt"] = pd.to_datetime(df["workout_date"], errors="coerce")
    df["month"] = df["workout_dt"].dt.to_period("M").astype("string")
    df["dow"] = df["workout_dt"].dt.day_name()

    # Logging delay in days (timestamp_date - workout_date)
    df["timestamp_date"] = df["timestamp"].dt.date
    df["log_delay_days"] = (
        pd.to_datetime(df["timestamp_date"], errors="coerce")
        - pd.to_datetime(df["workout_date"], errors="coerce")
    ).dt.days

    # Day of month
    df["dom"] = df["workout_dt"].dt.day

    # Optional: flag suspicious negative delays (logged before the workout date)
    df["delay_flag"] = np.where(df["log_delay_days"].isna(), pd.NA,
                                np.where(df["log_delay_days"] < 0, "negative_delay", ""))

    return df

df = clean(raw)
df.head(10)

,timestamp,name,workout_date,burnt_250_raw,name_raw,burnt_250,any_workout,workout_dt,month,dow,timestamp_date,log_delay_days,dom,delay_flag
1,2026-01-01 18:45:05,Eshwar,2026-01-01,Yes,Eshwar,True,True,2026-01-01,2026-01,Thursday,2026-01-01,0,1,
12,2026-01-03 00:08:01,Eshwar,2026-01-02,No,Eshwar,False,True,2026-01-02,2026-01,Friday,2026-01-03,1,2,
0,2026-01-01 15:30:34,Naveen Ganta,2026-01-01,Yes,Naveen Ganta,True,True,2026-01-01,2026-01,Thursday,2026-01-01,0,1,
11,2026-01-02 22:56:32,Naveen Ganta,2026-01-02,Yes,Naveen Ganta,True,True,2026-01-02,2026-01,Friday,2026-01-02,0,2,
6,2026-01-01 23:10:27,Pradyumna Ch.,2026-01-01,Yes,Pradyumna Ch.,True,True,2026-01-01,2026-01,Thursday,2026-01-01,0,1,
8,2026-01-02 20:13:34,Pradyumna Ch.,2026-01-02,Yes,Pradyumna Ch.,True,True,2026-01-02,2026-01,Friday,2026-01-02,0,2,
3,2026-01-01 21:27:05,Sandesh Ghanta,2026-01-01,Yes,Sandesh Ghanta,True,True,2026-01-01,2026-01,Thursday,2026-01-01,0,1,
2,2026-01-01 19:56:13,Srikar Gunisetty,2026-01-01,Yes,Srikar Gunisetty,True,True,2026-01-01,2026-01,Thursday,2026-01-01,0,1,
7,2026-01-02 19:57:22,Srivatsav Gunisetty,2026-01-02,Yes,Srivatsav Gunisetty,True,True,2026-01-02,2026-01,Friday,2026-01-02,0,2,
4,2026-01-01 21:59:46,Surya Chaitanya,2026-01-01,Yes,Surya Chaitanya,True,True,2026-01-01,2026-01,Thursday,2026-01-01,0,1,


## Monthly Leaderboard

Winner definition:
- For a given month, a person is a winner if they have at least 16 unique workout dates in that month
- Only count dates where burnt_250 == True

We also compute:
- Total unique workout days (regardless of calories), to reflect consistency without filtering to 250+

In [12]:
WINNER_CUTOFF = 16

def monthly_summary(df: pd.DataFrame) -> pd.DataFrame:
    # Qualifying days (>=250) per person-month
    qualifying = (
        df[df["burnt_250"] & df["any_workout"]]
        .groupby(["month", "name"])["workout_date"]
        .nunique()
        .reset_index(name="qualifying_days_250")
    )

    # Any workout days per person-month
    any_days = (
        df[df["any_workout"]]
        .groupby(["month", "name"])["workout_date"]
        .nunique()
        .reset_index(name="workout_days_any")
    )

    out = any_days.merge(qualifying, on=["month", "name"], how="left")
    out["qualifying_days_250"] = out["qualifying_days_250"].fillna(0).astype(int)
    out["is_winner"] = out["qualifying_days_250"] >= WINNER_CUTOFF

    return out.sort_values(["month", "is_winner", "qualifying_days_250", "workout_days_any"], ascending=[True, False, False, False])

ms = monthly_summary(df)
ms.head(20)

,month,name,workout_days_any,qualifying_days_250,is_winner
1,2026-01,Naveen Ganta,2,2,False
2,2026-01,Pradyumna Ch.,2,2,False
6,2026-01,Surya Chaitanya,2,2,False
7,2026-01,Vennela Chava,2,2,False
0,2026-01,Eshwar,2,1,False
3,2026-01,Sandesh Ghanta,1,1,False
4,2026-01,Srikar Gunisetty,1,1,False
5,2026-01,Srivatsav Gunisetty,1,1,False


## Day-of-week trends

This section answers:
- Overall: which weekday is most preferred for workout by everyboady?
- Overall: which weekday is least preferred for workout by everyboady?
- Overall: list of folks who prefers weekends to workout vs who prefers weekdays?
- Per person: what weekday is most preferred and least preferred

Important note:
- Day-of-week analysis uses workout_date (not timestamp).

In [18]:
WEEKDAY_ORDER = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
WEEKEND_DAYS = {"Saturday", "Sunday"}

# Safety: ensure required columns exist
required_cols = {"name", "dow", "workout_date", "any_workout"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns in df: {missing}. Found: {list(df.columns)}")

d = df[df["any_workout"]].copy()

# --- 1 & 2) Overall most/least preferred weekday ---
overall_dow = (
    d.groupby("dow")["workout_date"]
    .nunique()
    .reindex(WEEKDAY_ORDER)
    .fillna(0)
    .astype(int)
    .reset_index(name="unique_workout_days")
)

most_preferred_overall = overall_dow.loc[overall_dow["unique_workout_days"].idxmax()].to_dict()
least_preferred_overall = overall_dow.loc[overall_dow["unique_workout_days"].idxmin()].to_dict()

In [20]:
print("Overall weekday counts (unique workout dates across everyone):")
display(overall_dow)

Overall weekday counts (unique workout dates across everyone):


,dow,unique_workout_days
0,Monday,0
1,Tuesday,0
2,Wednesday,0
3,Thursday,1
4,Friday,1
5,Saturday,0
6,Sunday,0


In [22]:
print("\nOverall most preferred weekday:")
display(pd.DataFrame([most_preferred_overall]))


Overall most preferred weekday:


,dow,unique_workout_days
0,Thursday,1


In [24]:
print("\nOverall least preferred weekday:")
display(pd.DataFrame([least_preferred_overall]))


Overall least preferred weekday:


,dow,unique_workout_days
0,Monday,0


In [32]:
# --- 3) Weekend vs weekday preference per person ---
d = df[df["any_workout"]].copy()
d["is_weekend"] = d["dow"].isin(WEEKEND_DAYS).astype(bool)

weekend_weekday_pref = (
    d.groupby(["name", "is_weekend"])["workout_date"]
    .nunique()
    .reset_index(name="unique_days")
)

pref = weekend_weekday_pref.pivot(index="name", columns="is_weekend", values="unique_days")

# Always force BOTH columns to exist, and in a known order
pref = pref.reindex(columns=[False, True], fill_value=0).fillna(0)

# Rename to friendly column names
pref.columns = ["weekday_days", "weekend_days"]

pref["preference"] = np.select(
    [
        pref["weekend_days"] > pref["weekday_days"],
        pref["weekend_days"] < pref["weekday_days"],
    ],
    ["Weekend", "Weekday"],
    default="Tie"
)

pref_out = (
    pref.reset_index()
    .sort_values(["preference", "weekend_days", "weekday_days"], ascending=[True, False, False])
)

In [34]:
print("\nWeekend vs Weekday preference per person:")
display(pref_out)


Weekend vs Weekday preference per person:


,name,weekday_days,weekend_days,preference
0,Eshwar,2,0,Weekday
1,Naveen Ganta,2,0,Weekday
2,Pradyumna Ch.,2,0,Weekday
6,Surya Chaitanya,2,0,Weekday
7,Vennela Chava,2,0,Weekday
3,Sandesh Ghanta,1,0,Weekday
4,Srikar Gunisetty,1,0,Weekday
5,Srivatsav Gunisetty,1,0,Weekday


In [36]:
print("\nPeople who prefer weekends:")
display(pref_out[pref_out["preference"] == "Weekend"])


People who prefer weekends:


,name,weekday_days,weekend_days,preference


In [38]:
print("\nPeople who prefer weekdays:")
display(pref_out[pref_out["preference"] == "Weekday"])


People who prefer weekdays:


,name,weekday_days,weekend_days,preference
0,Eshwar,2,0,Weekday
1,Naveen Ganta,2,0,Weekday
2,Pradyumna Ch.,2,0,Weekday
6,Surya Chaitanya,2,0,Weekday
7,Vennela Chava,2,0,Weekday
3,Sandesh Ghanta,1,0,Weekday
4,Srikar Gunisetty,1,0,Weekday
5,Srivatsav Gunisetty,1,0,Weekday


In [40]:
print("\nPeople tied (same number of weekend and weekday workouts):")
display(pref_out[pref_out["preference"] == "Tie"])


People tied (same number of weekend and weekday workouts):


,name,weekday_days,weekend_days,preference


In [ ]:
# --- 4) Per person most/least preferred weekday (including zeros) ---
per_person_dow = (
    d.groupby(["name", "dow"])["workout_date"]
    .nunique()
    .reset_index(name="unique_days")
)

# Ensure every person has all 7 weekdays represented (fill missing with 0)
full_index = pd.MultiIndex.from_product(
    [per_person_dow["name"].unique(), WEEKDAY_ORDER],
    names=["name", "dow"]
)

per_person_dow_full = (
    per_person_dow
    .set_index(["name", "dow"])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

# Most preferred weekday per person (ties: returns all tied days)
max_days = per_person_dow_full.groupby("name")["unique_days"].transform("max")
most_pref = per_person_dow_full[per_person_dow_full["unique_days"] == max_days].copy()
most_pref = most_pref.rename(columns={"dow": "most_preferred_day", "unique_days": "most_preferred_days"})

# Least preferred weekday per person (ties: returns all tied days, often zeros)
min_days = per_person_dow_full.groupby("name")["unique_days"].transform("min")
least_pref = per_person_dow_full[per_person_dow_full["unique_days"] == min_days].copy()
least_pref = least_pref.rename(columns={"dow": "least_preferred_day", "unique_days": "least_preferred_days"})

In [ ]:
# Find each person's max workout count across weekdays
per_person_dow_full["max_days"] = per_person_dow_full.groupby("name")["unique_days"].transform("max")

# Keep only the weekdays tied for max (their preferred days)
preferred = per_person_dow_full[per_person_dow_full["unique_days"] == per_person_dow_full["max_days"]].copy()

# Collect preferred days into a Python list per person
preferred_days_list = (
    preferred.sort_values(["name", "dow"], key=lambda s: s.map({d:i for i,d in enumerate(WEEKDAY_ORDER)}))
    .groupby("name")["dow"]
    .apply(list)
    .reset_index(name="preferred_workout_days")
)

preferred_days_list

In [46]:
# Optional: single-row summary per person by joining tied days into comma-separated strings
most_joined = (
    most_pref.groupby("name")
    .agg(
        most_preferred_day=("most_preferred_day", lambda x: ", ".join(x)),
        most_preferred_days=("most_preferred_days", "max"),
    )
)

least_joined = (
    least_pref.groupby("name")
    .agg(
        least_preferred_day=("least_preferred_day", lambda x: ", ".join(x)),
        least_preferred_days=("least_preferred_days", "min"),
    )
)

dow_summary = (
    most_joined
    .merge(least_joined, left_index=True, right_index=True, how="left")
    .merge(pref[["weekday_days", "weekend_days", "preference"]], left_index=True, right_index=True, how="left")
    .reset_index()
    .rename(columns={"index": "name"})
    .sort_values("name")
)

print("\nPer-person day-of-week summary (single row per person):")
display(dow_summary)


Per-person day-of-week summary (single row per person):


,name,most_preferred_day,most_preferred_days,least_preferred_day,least_preferred_days,weekday_days,weekend_days,preference
0,Eshwar,"Thursday, Friday",1,"Monday, Tuesday, Wednesday, Saturday, Sunday",0,2,0,Weekday
1,Naveen Ganta,"Thursday, Friday",1,"Monday, Tuesday, Wednesday, Saturday, Sunday",0,2,0,Weekday
2,Pradyumna Ch.,"Thursday, Friday",1,"Monday, Tuesday, Wednesday, Saturday, Sunday",0,2,0,Weekday
3,Sandesh Ghanta,Thursday,1,"Monday, Tuesday, Wednesday, Friday, Saturday, ...",0,1,0,Weekday
4,Srikar Gunisetty,Thursday,1,"Monday, Tuesday, Wednesday, Friday, Saturday, ...",0,1,0,Weekday
5,Srivatsav Gunisetty,Friday,1,"Monday, Tuesday, Wednesday, Thursday, Saturday...",0,1,0,Weekday
6,Surya Chaitanya,"Thursday, Friday",1,"Monday, Tuesday, Wednesday, Saturday, Sunday",0,2,0,Weekday
7,Vennela Chava,"Thursday, Friday",1,"Monday, Tuesday, Wednesday, Saturday, Sunday",0,2,0,Weekday


## Logging delay analysis (timestamp vs workout date)

This section answers:
- Who logs workouts the latest (higher delay)?
- Median and mean delay per person per month
- Percent logged same-day vs 1+ days late

Interpretation:
- log_delay_days = 0 means logged on the same calendar day
- positive values mean logged late
- negative values usually indicate data entry issues (logged before the workout date) and should be reviewed

In [50]:
def logging_delay_summary(df: pd.DataFrame) -> pd.DataFrame:
    d = df[df["any_workout"]].copy()

    agg = (
        d.groupby(["month","name"])["log_delay_days"]
        .agg(
            submissions="count",
            mean_delay="mean",
            median_delay="median",
            p90_delay=lambda x: np.nanpercentile(x, 90) if len(x) else np.nan,
            same_day_rate=lambda x: float(np.mean(x == 0)) if len(x) else np.nan,
            late_rate=lambda x: float(np.mean(x > 0)) if len(x) else np.nan,
            negative_delay_rate=lambda x: float(np.mean(x < 0)) if len(x) else np.nan,
        )
        .reset_index()
    )

    # "Laziness" proxy score: prioritize median, then p90
    agg["lazy_score"] = agg["median_delay"].fillna(0) + 0.25 * agg["p90_delay"].fillna(0)
    return agg.sort_values(["month", "lazy_score"], ascending=[True, False])

ld = logging_delay_summary(df)
ld.head(20)

,month,name,submissions,mean_delay,median_delay,p90_delay,same_day_rate,late_rate,negative_delay_rate,lazy_score
0,2026-01,Eshwar,2,0.5,0.5,0.9,0.5,0.5,0.0,0.725
1,2026-01,Naveen Ganta,2,0.0,0.0,0.0,1.0,0.0,0.0,0.000
2,2026-01,Pradyumna Ch.,2,0.0,0.0,0.0,1.0,0.0,0.0,0.000
3,2026-01,Sandesh Ghanta,1,0.0,0.0,0.0,1.0,0.0,0.0,0.000
4,2026-01,Srikar Gunisetty,1,0.0,0.0,0.0,1.0,0.0,0.0,0.000
5,2026-01,Srivatsav Gunisetty,1,0.0,0.0,0.0,1.0,0.0,0.0,0.000
6,2026-01,Surya Chaitanya,2,0.0,0.0,0.0,1.0,0.0,0.0,0.000
7,2026-01,Vennela Chava,2,0.0,0.0,0.0,1.0,0.0,0.0,0.000


## Longest Streak

- Gives a list of top 3 candidates who maintained the longest streak of continuous work out dates in a given month.
- Order them by streak length and give me top 3 candidates

In [56]:
# -----------------------------
# Longest Streak (Top 3 per month)
# - Continuous streak of workout dates (any workout, regardless of calories)
# - Returns top 3 people per month ordered by longest streak length
# -----------------------------

def longest_streak_from_dates(dates) -> int:
    """
    dates: iterable of python date / datetime-like
    Returns length of longest consecutive-day streak.
    """
    if dates is None:
        return 0

    # Convert to unique sorted date ordinals
    ds = pd.to_datetime(pd.Series(list(dates)), errors="coerce").dropna().dt.date.unique()
    if len(ds) == 0:
        return 0

    ords = np.sort(pd.Series(ds).apply(lambda x: x.toordinal()).values)

    best = 1
    cur = 1
    for i in range(1, len(ords)):
        if ords[i] - ords[i - 1] == 1:
            cur += 1
            best = max(best, cur)
        else:
            cur = 1
    return int(best)

# Use workout_date only; compute streaks per person-month
streaks = (
    df[df["any_workout"] & df["workout_date"].notna()]
    .groupby(["month", "name"])["workout_date"]
    .apply(longest_streak_from_dates)
    .reset_index(name="longest_streak_days")
)

# Top 3 per month (ties: deterministic ordering by name)
top3_streaks_per_month = (
    streaks.sort_values(["month", "longest_streak_days", "name"], ascending=[True, False, True])
    .groupby("month")
    .head(3)
    .reset_index(drop=True)
)

top3_streaks_per_month

,month,name,longest_streak_days
0,2026-01,Eshwar,2
1,2026-01,Naveen Ganta,2
2,2026-01,Pradyumna Ch.,2


## Front-loading vs cramming (workouts early vs late in the month)

This section answers:
- Who tends to complete workouts earlier in the month vs later?
- Quantify behavior with a simple score

Approach:
- For each person-month, compute the average day-of-month for workout_date (lower means earlier)
- Compute share of workouts in:
  - first 10 days
  - last 10 days (or last third, depending on preference)

Interpretation:
- Higher "cram_score" means more workouts concentrated at the end of the month

In [ ]:
def frontload_cram_summary(df: pd.DataFrame) -> pd.DataFrame:
    d = df[df["any_workout"]].copy()

    # days in month for each month (for last-10-days logic)
    d["month_period"] = pd.to_datetime(d["month"] + "-01").dt.to_period("M")
    d["days_in_month"] = d["month_period"].dt.days_in_month

    def share_first10(x):
        return float(np.mean(x <= 10)) if len(x) else np.nan

    def share_last10(dom, dim):
        return float(np.mean(dom >= (dim - 9))) if len(dom) else np.nan

    rows = []
    for (m, name), g in d.groupby(["month", "name"]):
        dom = g["dom"].to_numpy()
        dim = int(g["days_in_month"].iloc[0])
        rows.append({
            "month": m,
            "name": name,
            "workout_days_any": g["workout_date"].nunique(),
            "avg_day_of_month": float(np.mean(dom)) if len(dom) else np.nan,
            "share_first_10_days": share_first10(dom),
            "share_last_10_days": share_last10(dom, dim),
        })

    out = pd.DataFrame(rows)
    out["cram_score"] = out["share_last_10_days"].fillna(0) - out["share_first_10_days"].fillna(0)
    return out.sort_values(["month", "cram_score"], ascending=[True, False])

fc = frontload_cram_summary(df)
fc.head(20)

## 8) Streaks and inconsistency

We compute:
- Longest streak of consecutive workout dates (any workout)
- Longest streak of consecutive qualifying dates (burnt_250 == True)
- An inconsistency proxy based on the variability of gaps between workouts

Notes:
- Streaks are computed within each month
- If you want streaks across months (continuous), we can change the grouping logic

In [ ]:
def longest_streak(dates: list) -> int:
    if not dates:
        return 0
    # Unique and sorted
    ds = sorted(set(pd.to_datetime(dates).date))
    best = cur = 1
    for i in range(1, len(ds)):
        if (pd.to_datetime(ds[i]) - pd.to_datetime(ds[i-1])).days == 1:
            cur += 1
            best = max(best, cur)
        else:
            cur = 1
    return best

def streaks_inconsistency(df: pd.DataFrame) -> pd.DataFrame:
    d = df[df["any_workout"]].copy()

    rows = []
    for (m, name), g in d.groupby(["month", "name"]):
        any_dates = g["workout_date"].dropna().tolist()
        q_dates = g.loc[g["burnt_250"], "workout_date"].dropna().tolist()

        # gap variability as inconsistency proxy
        sds = sorted(set(pd.to_datetime(any_dates)))
        gaps = np.diff([x.toordinal() for x in sds]) if len(sds) > 1 else np.array([])
        gap_std = float(np.std(gaps)) if len(gaps) else 0.0
        gap_mean = float(np.mean(gaps)) if len(gaps) else 0.0
        gap_cv = float(gap_std / gap_mean) if gap_mean > 0 else 0.0

        rows.append({
            "month": m,
            "name": name,
            "longest_streak_any": longest_streak(any_dates),
            "longest_streak_250": longest_streak(q_dates),
            "gap_mean_days": gap_mean,
            "gap_cv": gap_cv,  # higher means more irregular spacing
        })

    out = pd.DataFrame(rows)
    return out.sort_values(["month", "longest_streak_any"], ascending=[True, False])

si = streaks_inconsistency(df)
si.head(20)

## 9) Barely missed and leaderboards

Barely missed:
- People with 14 or 15 qualifying days (>=250) in the month

Leaderboards:
- Most qualifying days
- Most total workout days (any)
- Strongest streak
- Most punctual logger (lowest median delay)
- Most cramming (highest cram_score)

In [ ]:
def barely_missed(ms: pd.DataFrame, low=14, high=15) -> pd.DataFrame:
    return ms[(ms["qualifying_days_250"] >= low) & (ms["qualifying_days_250"] <= high)].sort_values(["month","qualifying_days_250"], ascending=[True, False])

bm = barely_missed(ms)
bm.head(20)

## 10) Month-over-month trends

This section provides:
- Total participants per month
- Total workouts per month
- Average qualifying days among participants
- Winner count per month

These help you understand engagement growth and seasonality.

In [ ]:
def mom_trends(df: pd.DataFrame, ms: pd.DataFrame) -> dict:
    participants = (
        df[df["any_workout"]]
        .groupby("month")["name"]
        .nunique()
        .reset_index(name="participants")
    )

    total_workouts_any = (
        df[df["any_workout"]]
        .groupby("month")["workout_date"]
        .nunique()
        .reset_index(name="unique_workout_dates_logged")
    )

    avg_qualifying = (
        ms.groupby("month")["qualifying_days_250"]
        .mean()
        .reset_index(name="avg_qualifying_days_250_per_person")
    )

    winners = (
        ms.groupby("month")["is_winner"]
        .sum()
        .reset_index(name="winner_count")
    )

    out = participants.merge(total_workouts_any, on="month", how="left").merge(avg_qualifying, on="month", how="left").merge(winners, on="month", how="left")
    out = out.sort_values("month")

    return {"summary": out}

mom = mom_trends(df, ms)["summary"]
mom

## 11) Gamification awards (example)

You can announce awards per month to keep people motivated.
This section creates a simple set of awards per month:

- Winner(s): qualifying_days_250 >= 16
- Most consistent: max workout_days_any
- Iron streak: max longest_streak_any
- Punctual logger: min median_delay (among those with enough submissions)
- Early bird: lowest avg_day_of_month
- The close call: max qualifying_days_250 among non-winners

You can customize thresholds and award names.

In [ ]:
def build_awards(ms: pd.DataFrame, ld: pd.DataFrame, fc: pd.DataFrame, si: pd.DataFrame) -> pd.DataFrame:
    # Merge key features
    base = ms.merge(si, on=["month","name"], how="left").merge(fc[["month","name","avg_day_of_month","cram_score"]], on=["month","name"], how="left")
    base = base.merge(ld[["month","name","median_delay","submissions"]], on=["month","name"], how="left")

    awards = []
    for m, g in base.groupby("month"):
        # Winners list
        winners = g[g["is_winner"]].sort_values(["qualifying_days_250","workout_days_any"], ascending=[False, False])
        winners_list = ", ".join(winners["name"].tolist()) if len(winners) else ""

        # Most consistent
        mc = g.sort_values(["workout_days_any","qualifying_days_250"], ascending=[False, False]).head(1)

        # Iron streak
        streak = g.sort_values(["longest_streak_any","workout_days_any"], ascending=[False, False]).head(1)

        # Punctual logger (require at least 3 submissions to reduce noise)
        punctual_pool = g[g["submissions"].fillna(0) >= 3]
        punctual = punctual_pool.sort_values(["median_delay","submissions"], ascending=[True, False]).head(1) if len(punctual_pool) else g.sort_values(["median_delay"], ascending=[True]).head(1)

        # Early bird (lower avg day)
        early = g.sort_values(["avg_day_of_month"], ascending=[True]).head(1)

        # Close call
        close = g[~g["is_winner"]].sort_values(["qualifying_days_250","workout_days_any"], ascending=[False, False]).head(1)

        awards.append({
            "month": m,
            "winners": winners_list,
            "most_consistent": mc["name"].iloc[0] if len(mc) else "",
            "most_consistent_days": int(mc["workout_days_any"].iloc[0]) if len(mc) else 0,
            "iron_streak": streak["name"].iloc[0] if len(streak) else "",
            "iron_streak_len": int(streak["longest_streak_any"].iloc[0]) if len(streak) else 0,
            "punctual_logger": punctual["name"].iloc[0] if len(punctual) else "",
            "punctual_median_delay": float(punctual["median_delay"].iloc[0]) if len(punctual) else np.nan,
            "early_bird": early["name"].iloc[0] if len(early) else "",
            "early_bird_avg_dom": float(early["avg_day_of_month"].iloc[0]) if len(early) else np.nan,
            "close_call": close["name"].iloc[0] if len(close) else "",
            "close_call_qualifying_days": int(close["qualifying_days_250"].iloc[0]) if len(close) else 0,
        })

    return pd.DataFrame(awards).sort_values("month")

awards = build_awards(ms, ld, fc, si)
awards

## 12) Simple visualization examples

Keep these lightweight and readable. A few charts that work well:
- Monthly qualifying-day leaderboard (bar chart)
- Day-of-week distribution
- Logging delay distribution

You can expand or move these to the Streamlit app.

In [ ]:
def plot_month_leaderboard(ms: pd.DataFrame, month: str, top_n=15):
    g = ms[ms["month"] == month].sort_values("qualifying_days_250", ascending=False).head(top_n)
    plt.figure()
    plt.bar(g["name"], g["qualifying_days_250"])
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Qualifying days (>=250 calories)")
    plt.title(f"Qualifying days leaderboard - {month}")
    plt.tight_layout()
    plt.show()

if example_month:
    plot_month_leaderboard(ms, example_month, top_n=15)